# Docker

对应 `codes/index.html` 的 Docker 模块：镜像、卷、端口、GPU、Compose。目标只有一个——把**环境**和**数据**分开：环境装进镜像，数据留在宿主机。

笔记本在 `codes/docker/qa.ipynb`。需要读写文件时，工作空间就是这个目录，Dockerfile 都在 `app/`。

先运行下一格，得到 `ROOT` 和 `run()`。每题只改 `# 作答` 下面的代码。做完自己跑通即可，先不要对答案。

本机是 macOS + Docker Desktop，没有 NVIDIA GPU。第 5 题会跑不通，那是题的一部分，不是你的环境坏了。

In [ ]:
import shutil
import subprocess
from pathlib import Path


def lab_root() -> Path:
    """qa.ipynb 所在目录。"""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if folder.name == "docker" and (folder / "qa.ipynb").is_file():
            return folder
        candidate = folder / "codes" / "docker"
        if (candidate / "qa.ipynb").is_file():
            return candidate
    return here


def run(cmd: str, cwd: Path | None = None) -> str:
    """在 ROOT（或指定目录）跑一条 shell 命令，回显命令和输出，返回 stdout。"""
    print(f"$ {cmd}")
    done = subprocess.run(cmd, shell=True, cwd=cwd or ROOT, text=True, capture_output=True)
    print((done.stdout + done.stderr).rstrip())
    return done.stdout


ROOT = lab_root()
print("ROOT:", ROOT)
print("docker:", shutil.which("docker"))
run("docker compose version")

## 1. 第一次 build

`app/Dockerfile` 把 numpy 和 `main.py` 装进镜像。先读一遍文件，然后：

1. `docker build -t docker-lab:1 app`
2. `docker run --rm docker-lab:1`
3. 在宿主机跑一次 `uname -m`

回答：容器里的 `arch` 和宿主机的输出一样吗？「同一镜像到处能跑」保证了什么、又没保证什么？

In [ ]:
# 作答：build 并 run

## 2. 层缓存：先装依赖，再拷代码

只改 `app/main.py` 的一句 `print`，再 build 一次 `docker-lab:1`。看输出里哪几步是 `CACHED`。

再读 `app/Dockerfile.bad`：它把 `COPY . .` 放在 `pip install` 前面。用它 build 成 `docker-lab:bad`，再改一次 `main.py`，再 build 一次。

```bash
docker build -f app/Dockerfile.bad -t docker-lab:bad app
```

回答：两次 build 各重跑了哪一步？为什么 `Dockerfile.bad` 改一行代码就要重装 numpy？

In [ ]:
# 作答：两次 build，对比 CACHED

## 3. 卷：模型和数据别跟着容器消失

`app/write_artifact.py` 往 `/artifacts/embeddings.npy` 写一个矩阵。

1. 不挂卷跑一次，并在同一个容器里 `ls /artifacts`。
2. 再起一个新容器，`ls /artifacts`。
3. `mkdir -p artifacts`，加 `-v "$PWD/artifacts:/artifacts"` 跑一次，然后在宿主机 `ls artifacts`。

回答：`--rm` 删掉的是什么？挂载之后，文件到底存在哪？

In [ ]:
# 作答：先不挂卷，再挂卷

## 4. 端口：容器里的 8000 不等于你的 8000

在容器里跑 `python -m http.server 8000`，把宿主机 `8888` 映射到容器 `8000`，用 curl 访问 `http://localhost:8888/`。加 `-d` 让容器在后台跑，加 `--name lab-http` 方便停掉。

回答：`-p 8888:8000` 哪一边是宿主机？换成 Jupyter，命令要改哪两处？

In [ ]:
# 作答：起服务、访问、停掉

## 5. GPU：本机没有，就学会看报错

用刚 build 的镜像试一次 `--gpus all`：

```bash
docker run --rm --gpus all docker-lab:1 python -c "print('gpu ok')"
```

它会在启动阶段失败。把报错抄下来。

回答：

1. 报错来自 Docker 客户端、守护进程，还是容器内部？
2. 同样的 Dockerfile 拿到 Linux + NVIDIA 的机器上，还要装什么，`--gpus all` 才会生效？
3. 机器没有 GPU 时，`pip install torch --index-url .../cu124` 这种写法会留下什么后果？

In [ ]:
# 作答：跑一次，记录报错

## 6. Compose：两个服务，一个网络

读 `app/compose.yaml`：`lab` 用刚 build 的镜像，`cache` 是 redis，数据挂在命名卷 `cache_data` 上。

1. 在 `app/` 里 `docker compose up -d`，看两个容器。
2. 从 `lab` 里解析 `cache` 这个主机名（用 `socket.gethostbyname`）。
3. 往 redis 写一个键；`docker compose down` 后 `up -d`，再读。
4. `docker compose down -v` 后 `up -d`，再读。

回答：`down` 和 `down -v` 差在哪？为什么 `lab` 里可以直接写 `cache:6379`？

In [ ]:
# 作答：up、解析主机名、写键、down 与 down -v

## 跑完自检

`./check.sh` 会把上面 6 题的关键行为跑一遍并断言：镜像能跑、层顺序对、卷能持久、Compose 能连通。

第一次要拉 `python:3.12-slim` 和 `redis:7-alpine`，全量约 1~2 分钟。